> **2026** — local-first biology (Biopython/PDB/pandas); optional paid LLM only where noted. See `UPDATE_2026.md`.

# Chapter 7 — Biology Evidence Workflow (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2007.%20LangChain%20for%20Biology/LC4LSH_Chapter_7_Biology_Evidence_Workflow.ipynb)

**Learning objectives**
- Separate annotation / measurement / prediction / hypothesis
- Attach explicit confidence and provenance to each claim
- Apply confidence gates and abstain when evidence is thin
- (Optional) Use a gated LLM only to format, never to fabricate

> Runtime: ~5 min (local; optional paid LLM)  
> Cost: free (no LLM needed)  
> Data: small built-in evidence records


## Environment setup


### Secrets (optional LLM only)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

# These notebooks run locally (Biopython/pandas); a paid LLM is OPTIONAL for narrative only.
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LS_OPENAI_API_KEY", "sk-...")
print("Optional LLM provider:", API_KEY_PROVIDER, "(analysis runs without it)")
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q biopython "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4" "langchain==1.0.0" "langchain-openai==1.0.0" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter7-evidence-workflow"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local-first)")


## The core rule

**Do not infer function, pathogenicity, or biological mechanism from sequence/cluster patterns alone.** Every claim must carry a type (annotation / measurement / prediction / hypothesis), a confidence, and a provenance. When evidence is thin, **abstain**.


## 1. A typed evidence record


In [ ]:
from dataclasses import dataclass, field
from typing import Literal

ClaimType = Literal["annotation", "measurement", "prediction", "hypothesis"]
CONF_RANK = {"low": 0, "medium": 1, "high": 2}

@dataclass
class Evidence:
    subject: str
    claim: str
    ctype: ClaimType
    confidence: str            # low | medium | high
    source: str                # provenance
    support: list = field(default_factory=list)

    def rank(self):
        return CONF_RANK.get(self.confidence, 0)

ev = [
    Evidence("GENE5", "kinase domain present", "annotation", "high", "InterPro:IPR000719"),
    Evidence("GENE5", "upregulated in group A", "measurement", "medium", "omics-qc-viz (toy)"),
    Evidence("GENE5", "involved in cell-cycle control", "hypothesis", "low", "none"),
]
for e in ev:
    print(f"[{e.ctype:12s}] ({e.confidence:6s}) {e.claim}  <- {e.source}")


## 2. Gate: keep only supported claims


In [ ]:
def gate(evidence, min_conf="medium", allowed=("annotation", "measurement")):
    kept, dropped = [], []
    for e in evidence:
        if e.ctype in allowed and e.rank() >= CONF_RANK[min_conf]:
            kept.append(e)
        else:
            dropped.append(e)
    return kept, dropped

kept, dropped = gate(ev)
print("KEPT:")
for e in kept: print("  ", e.ctype, "-", e.claim)
print("ABSTAINED/DROPPED:")
for e in dropped: print("  ", e.ctype, "-", e.claim, f"({e.confidence})")


## 3. Abstention when evidence is thin


In [ ]:
def answer(subject, evidence):
    kept, _ = gate(evidence)
    if not kept:
        return f"ABSTAIN: no sufficiently-supported annotation/measurement for {subject}."
    lines = [f"Supported statements about {subject}:"]
    for e in kept:
        lines.append(f" - [{e.ctype}/{e.confidence}] {e.claim} (source: {e.source})")
    lines.append("(Hypotheses and low-confidence predictions are withheld by the gate.)")
    return "\n".join(lines)

print(answer("GENE5", ev))
print()
print(answer("GENE99", []))  # abstains


## 4. Retrieval-style provenance (toy)


In [ ]:
DOCS = {
    "IPR000719": "Protein kinase domain (InterPro).",
    "PMID:111": "GENE5 transcript elevated in condition A (small n).",
}
def cite(evidence):
    out = []
    for e in evidence:
        ref = e.source.split(":")[0]
        out.append((e.claim, DOCS.get(e.source, DOCS.get(ref, "no source text"))))
    return out

for claim, src in cite(kept):
    print(f"- {claim}
    evidence: {src}")


## 5. Optional: LLM formats (never fabricates)


In [ ]:
USE_LLM = False
if USE_LLM and kept:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    payload = "\n".join(f"[{e.ctype}/{e.confidence}] {e.claim} (source: {e.source})" for e in kept)
    prompt = ("Rewrite the following SUPPORTED statements as a short report. "
              "Do not add any new claim. Keep the confidence labels.\n" + payload)
    print(llm.invoke(prompt).content)
else:
    print("LLM formatting OFF (USE_LLM=False or nothing kept). The gated list above is the answer.")


## Limitations & safety notes

- Toy example; real workflows need curated databases and richer provenance.
- The gate encodes a policy (which types/confidences count); tune it per application.
- The LLM only reformats already-gated claims — it must not introduce new facts.
- Local/free; optional paid LLM clearly marked.


In [ ]:
# Cleanup
import gc
for _v in ("records", "df", "model", "llm", "structure", "X"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why abstain instead of guessing?</summary>A wrong confident answer is worse than no answer in biology; abstention preserves trust.</details>

<details><summary>Why separate prediction from measurement?</summary>They carry different error models; measurements are observed, predictions are model outputs.</details>

<details><summary>Why keep provenance on every claim?</summary>So each statement can be traced back to a database entry, experiment, or model version.</details>

### Tasks
- **Task A** - Add a numeric confidence score and a per-claim threshold instead of the low/medium/high rank.
- **Task B** - Wire the gate to a real annotation source (e.g., InterPro/UniProt) with cached lookups.
- **Task C** - Add a contradiction check: flag when two kept claims conflict and require manual review.
- **Task D** - Turn the LLM formatter on with a strict no-new-claims prompt and verify it never exceeds the gated list.
